# Olist customer and region EDA

This notebook builds the first Olist preparation artifact and then narrows the analysis to the largest customer region, which is SP, before preparing a customer-level clustering dataset.

The workflow follows the project pattern used elsewhere in the repo: keep the raw files under `data/raw/olist`, prepare derived tables under `data/prepared/olist`, and keep the narrative and code in separate cells.


In [33]:
from pathlib import Path
import pandas as pd

from data_prep.prepare import prepare_dataset

workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

raw_dir = (workspace_root / "data" / "raw" / "olist").resolve()
prepared_dir = (workspace_root / "data" / "prepared" / "olist").resolve()
prepared_dir.mkdir(parents=True, exist_ok=True)

print("Workspace root:", workspace_root)
print("Raw Olist directory:", raw_dir)
print("Prepared Olist directory:", prepared_dir)


def prepare_olist_dataset(dataframe, *, prepared_dataset_dir, dataset_name="olist", output_format="csv", metadata=None):
    """Notebook-local Olist preparation logic using only the final dictionary subset."""
    cleaned = dataframe.copy()
    required_columns = [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_item_id",
        "product_id",
        "price",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    ]
    missing = [col for col in required_columns if col not in cleaned.columns]
    if missing:
        raise ValueError(f"Olist dataset is missing required columns: {missing}")

    prepared = cleaned[required_columns].copy()
    prepared["order_purchase_timestamp"] = pd.to_datetime(prepared["order_purchase_timestamp"], errors="coerce")

    result_metadata = {
        "source": "olist",
        "preparation_logic": "retain_final_dictionary_subset",
        "selected_fields": required_columns,
    }
    if metadata:
        result_metadata.update(metadata)

    return prepare_dataset(
        dataset_name=dataset_name,
        representation="tabular",
        task_characterization="clustering",
        staging_dir="./data/raw/olist",
        prepared_dataset_dir=prepared_dataset_dir,
        data=prepared,
        output_format=output_format,
        filename=f"{dataset_name}_prepared",
        metadata=result_metadata,
    )


Workspace root: /home/rajiv/programming/kmds-dataset-util
Raw Olist directory: /home/rajiv/programming/kmds-dataset-util/data/raw/olist
Prepared Olist directory: /home/rajiv/programming/kmds-dataset-util/data/prepared/olist


## 1. Prepare the daily orders dataset

The first preparation step combines the orders, order items, and customer metadata into a single daily-order table. The daily-order table is the reusable base dataset for downstream revenue and region analysis.


In [34]:
orders_path = raw_dir / "olist_orders_dataset.csv"
items_path = raw_dir / "olist_order_items_dataset.csv"
customers_path = raw_dir / "olist_customers_dataset.csv"

df_orders = pd.read_csv(orders_path)
df_order_items = pd.read_csv(items_path)
df_customers = pd.read_csv(customers_path)

# Start with the transactional base table, then attach customer metadata to each order.
df_daily_orders = pd.merge(df_orders, df_order_items, on="order_id", how="inner")

customer_cols = ["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state"]
df_customers = df_customers[customer_cols].copy()
df_daily_orders = df_daily_orders.merge(df_customers, on="customer_id", how="left")

cols_needed = [
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_item_id",
    "product_id",
    "price",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
]

df_daily_orders = df_daily_orders[cols_needed].copy()
df_daily_orders["order_purchase_timestamp"] = pd.to_datetime(df_daily_orders["order_purchase_timestamp"])

product_counts = df_daily_orders.groupby("product_id")["order_id"].nunique().reset_index(name="n_orders")
freq_threshold = product_counts["n_orders"].quantile(0.8)
product_counts["freq_purch_prod"] = product_counts["n_orders"] >= freq_threshold
df_daily_orders = df_daily_orders.merge(product_counts[["product_id", "freq_purch_prod"]], on="product_id", how="left")
df_daily_orders = df_daily_orders.sort_values("order_id").reset_index(drop=True)

prepare_result = prepare_olist_dataset(
    df_daily_orders,
    prepared_dataset_dir=prepared_dir,
    dataset_name="olist",
    output_format="csv",
    metadata={"source_data": ["olist_orders_dataset.csv", "olist_order_items_dataset.csv", "olist_customers_dataset.csv"]},
)

prepared_path = Path(prepare_result["output_path"])
print("Prepared file:", prepared_path)
print(df_daily_orders.head())
print("\nPrepare result:", prepare_result)


Prepared file: /home/rajiv/programming/kmds-dataset-util/data/prepared/olist/olist_prepared.csv
                           order_id                       customer_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  3ce436f183e68e07877b285a838db11a   
1  00018f77f2f0320c557190d7a144bdd3  f6dd3ec061db4e3987629fe6b26e5cce   
2  000229ec398224ef6ca0657da4fc703e  6489ae5e4333f3693df5ad4372dab6d3   
3  00024acbcdf0a6daa1e931b038114c75  d4eb9395c8c0431ee92fce09860c5a06   
4  00042b26cf59d7ce69dfabb4e55b4fd9  58dbd0b2d70206bf40e62cd34e84d795   

  order_purchase_timestamp  order_item_id                        product_id  \
0      2017-09-13 08:59:02              1  4244733e06e7ecb4970a6e2683c13e61   
1      2017-04-26 10:53:06              1  e5f2d52b802189ee658865ca93d83a8f   
2      2018-01-14 14:33:31              1  c777355d18b72b67abbeef9df44fd0fd   
3      2018-08-08 10:00:35              1  7634da152a4610f1595efa32f14722fc   
4      2017-02-04 13:57:51              1  ac6c3623068f30de03045865e4e

## 2. Compare regional customer demand and justify SP as the prime region

The code below filters to the key states used in the analysis, strips out the 2016 snapshot that is not part of the comparison window, and compares total revenue by state. This makes the business case for focusing on SP before clustering customers in a region-specific model.


In [35]:
prepared_orders_path = prepared_dir / "olist_daily_orders_prepared.csv"
df = pd.read_csv(prepared_orders_path)
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

df["year"] = df["order_purchase_timestamp"].dt.year
df["month"] = df["order_purchase_timestamp"].dt.month
df["woy"] = df["order_purchase_timestamp"].dt.isocalendar().week

geo_filter = df["customer_state"].isin(["SP", "RJ", "MG"])
df = df[geo_filter].copy().reset_index(drop=True)
df = df[df["year"] != 2016].copy().reset_index(drop=True)

state_summary = (
    df.groupby(["customer_state", "year"])[["price"]]
    .sum()
    .reset_index()
    .rename(columns={"price": "total_revenue"})
)

largest_state = (
    state_summary.groupby("customer_state")["total_revenue"]
    .sum()
    .sort_values(ascending=False)
)
largest_state_name = largest_state.index[0]

print("Revenue by state and year:\n", state_summary.sort_values(["customer_state", "year"]).to_string(index=False))
print("\nLargest revenue state overall:", largest_state_name)
print("Revenue totals by state:\n", largest_state.to_string())

if largest_state_name != "SP":
    raise ValueError(f"Expected SP to be the largest region, but it was {largest_state_name}")

df_sp = df[df["customer_state"] == "SP"].copy().reset_index(drop=True)
print("SP rows:", len(df_sp))


Revenue by state and year:
 customer_state  year  total_revenue
            MG  2017      723229.03
            MG  2018      857267.79
            RJ  2017      906199.81
            RJ  2018      906646.41
            SP  2017     2212487.00
            SP  2018     2975757.23

Largest revenue state overall: SP
Revenue totals by state:
 customer_state
SP    5188244.23
RJ    1812846.22
MG    1580496.82
SP rows: 47323


## 3. Prepare the SP weekly product-sales clustering dataset

This step creates the clustering dataset by week. Each row represents one week of the year in SP, and each column represents a product. The value is the total weekly sales for that product. This is the dataset to be clustered.


In [36]:
# Restrict to the SP region and the relevant year window used in the analysis
sp_2017 = df_sp[df_sp["year"] == 2017].copy().reset_index(drop=True)
sp_2018 = df_sp[df_sp["year"] == 2018].copy().reset_index(drop=True)

# Each row is a week of the year; each column is a product; values are weekly sales totals.
df_SP_weekly_FPS_2017 = pd.pivot_table(
    sp_2017,
    index="woy",
    values="price",
    columns="product_id",
    aggfunc="sum",
    fill_value=0,
)

df_SP_weekly_FPS_2018 = pd.pivot_table(
    sp_2018,
    index="woy",
    values="price",
    columns="product_id",
    aggfunc="sum",
    fill_value=0,
)

# Save the clustering-ready matrices to the prepared directory.
sp_weekly_2017_path = prepared_dir / "SP_weekly_FPS_2017.csv"
sp_weekly_2018_path = prepared_dir / "SP_weekly_FPS_2018.csv"

df_SP_weekly_FPS_2017.to_csv(sp_weekly_2017_path)
df_SP_weekly_FPS_2018.to_csv(sp_weekly_2018_path)

print("SP weekly product-sales matrix (2017):", sp_weekly_2017_path)
print("Number of rows in the 2017 dataset:", df_SP_weekly_FPS_2017.shape[0])
print("Number of rows in the 2018 dataset:", df_SP_weekly_FPS_2018.shape[0])
print("\nSP weekly product-sales matrix (2018):", sp_weekly_2018_path)
print(df_SP_weekly_FPS_2018.shape[0])


SP weekly product-sales matrix (2017): /home/rajiv/programming/kmds-dataset-util/data/prepared/olist/SP_weekly_FPS_2017.csv
Number of rows in the 2017 dataset: 52
Number of rows in the 2018 dataset: 36

SP weekly product-sales matrix (2018): /home/rajiv/programming/kmds-dataset-util/data/prepared/olist/SP_weekly_FPS_2018.csv
36
